**Paper figures moved.** Figs. 2 to 7 of the MTAP paper are generated by `figures/make_all.py` (`--only fig02 ... fig07`), see `figures/README.md`. The notebook keeps the exploratory plots only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.cluster import KMeans
from rank import rank_qoe_3_classes, rank_impact_3_classes, rank_str_to_int_mapping_3_classes

In [ ]:
# Configuration for plot styling - modify these values to change all plots
PLOT_CONFIG = {
    'base_font_size': 14,
    'axis_label_size': 20,
    'title_size': 18,
    'legend_size': 20,
    'tick_label_size': 16,
    'figure_size': (12, 6),
    'marker_size': 80,
    'grid_alpha': 0.7,
    'dpi': 300,
    'output_dir': 'data_exploration_plots/'
}

In [ ]:
# Rating Quality Distortion
# 5 Excellent Imperceptible
# 4 Good Just perceptible, but not annoying
# 3 Fair Perceptible and slightly annoying
# 2 Poor Annoying, but not objectionable
# 1 Bad Very annoying and objectionable
def rank_qoe(ping):
    if ping <= 80:
        return 'Excellent'
    elif 80 < ping <= 120:
        return 'Good'
    elif 120 < ping <= 160:
        return 'Fair'
    elif 160 < ping < 200:
        return 'Poor'
    elif 200 >= ping:
        return 'Bad'
    return 'Error'

def rank_impact(impact):
    if impact <= 80:
        return 'Bad'
    elif 80 < impact <= 120:
        return 'Poor'
    elif 120 < impact <= 160:
        return 'Fair'
    elif 160 < impact < 200:
        return 'Good'
    elif 200 >= impact:
        return 'Excellent'
    return 'Error'

rank_str_to_int_mapping = {
    'Excellent': 5,
    'Good': 4,
    'Fair': 3,
    'Poor': 2,
    'Bad': 1
}

In [ ]:
print("Default DPI:", plt.rcParams['figure.dpi'])
plt.rcParams['figure.dpi'] = 300

In [ ]:
# window_path = '/Users/<user>/PhD/github/qoe-research/exp/17-May-2024/window.csv'
window_path = '/home/jony/git/Gaming-qoe-research/output/window_10_1754904064.csv'
df = pd.read_csv(window_path)

# table_path = '/Users/<user>/PhD/github/qoe-research/exp/17-May-2024/summary.csv'
table_path = '/home/jony/git/Gaming-qoe-research/COD-QoE/COD-experiments-metadata.csv'
exp_df = pd.read_csv(table_path)

In [ ]:
len(df['exp_id'])

In [ ]:
valid_exp_list = list(exp_df['exp_id'].unique())
# fitler out invalid exp_id from df
df = df[df['exp_id'].isin(valid_exp_list)]
len(df['exp_id'])

In [ ]:
exp_df['label'] = exp_df['avg_ping_rate'].apply(rank_qoe_3_classes)
df = df.merge(exp_df[['exp_id', 'label']], on='exp_id', how='left')

In [ ]:
exp_df.groupby(by=['label']).agg({
    'accuracy': ['median', 'mean'],
    'score': ['median', 'mean'],
    'self_rate_qoe_score': ['median', 'mean'],
    'kill': ['median', 'mean'],
    'death': ['median', 'mean', 'count'],
}).sort_values(by='label').dropna()

In [ ]:
df.self_rate_qoe_score.value_counts()

In [ ]:
impact_df = df.groupby(by=['exp_id', 'ping', 'impact']).agg({
    'src2dst_pps': 'median',
    'dst2src_pps': 'median',
    'src2dst_avg_pkt_size': 'median',
    'dst2src_avg_pkt_size': 'median',
}).sort_values(by='ping').reset_index()

In [ ]:
analysis_df = df.groupby(by=['exp_id', 'ping', 'label']).agg({
    'src2dst_pps': 'median',
    'dst2src_pps': 'median',
    'src2dst_avg_pkt_size': 'median',
    'dst2src_avg_pkt_size': 'median',
}).sort_values(by='ping').reset_index()

In [ ]:
exp7_df = df[df['exp_id'] == 7]
# exp7_df['src2dst_bytes'] = exp7_df['src2dst_pps'] * exp7_df['src2dst_avg_pkt_size']
# exp7_df['dst2src_bytes'] = exp7_df['dst2src_pps'] * exp7_df['dst2src_avg_pkt_size']
# print('src2dst_bytes', exp7_df['src2dst_bytes'].sum())
# print('src2dst_bytes', exp7_df['dst2src_bytes'].sum())
# print('duration', len(exp7_df))

In [ ]:
pd.DataFrame(analysis_df.groupby('self_rate_qoe_score')['ping'])

In [ ]:
analysis_df.label.value_counts()

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.ylabel('Subjective QoE Rating', fontsize=PLOT_CONFIG['axis_label_size'])
plt.xlabel('Ping Rate', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('Subjective QoE Rating vs. Ping Rate', fontsize=PLOT_CONFIG['title_size'])

# Use the 3-class system instead of the old 5-class system
data_to_plot = [group["ping"].tolist() for _, group in analysis_df.groupby('label')]

# Create the box chart
plt.boxplot(data_to_plot)

# Set labels and title
plt.xlabel('QoE Class (3-class system)', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('Ping Rate', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('QoE Classes vs. Ping Rate', fontsize=PLOT_CONFIG['title_size'])

# Set x-axis labels to show the 3 QoE classes
qoe_classes = sorted(analysis_df['label'].unique())
plt.xticks(range(1, len(qoe_classes) + 1), qoe_classes)

plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}QoE_Classes_vs_Ping_Rate_Boxplot.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.ylabel('Subjective QoE Rating', fontsize=PLOT_CONFIG['axis_label_size'])
plt.xlabel('Ping Rate', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('Subjective QoE Rating vs. Ping Rate', fontsize=PLOT_CONFIG['title_size'])

for i, row in analysis_df.iterrows():
    y1, x, label = row['self_rate_qoe_score'], row['ping'], row['exp_id']
    plt.yticks(range(1,6))
    plt.scatter(x, y1, c='red', marker='o', s=PLOT_CONFIG['marker_size'])
    # plt.scatter(x, y2, c='blue', marker='o')

plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}Subjective_QoE_vs_Ping_Rate.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.xlabel('Subjective QoE Rating', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('Ping Rate', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('Ping Rate vs. Subjective QoE Rating', fontsize=PLOT_CONFIG['title_size'])

for i, row in analysis_df.iterrows():
    x, y1, label = row['self_rate_qoe_score'], row['ping'], row['exp_id']
    plt.xticks(range(1,6))
    plt.scatter(x, y1, c='red', marker='o', s=PLOT_CONFIG['marker_size'])
    # plt.scatter(x, y2, c='blue', marker='o')

plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}Ping_Rate_vs_Subjective_QoE.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.xlabel('ACR', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('PPS', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('PPS vs. ACR', fontsize=PLOT_CONFIG['title_size'])

for i, row in analysis_df.iterrows():
    x, y1, y2, label = row['self_rate_qoe_score'], row['src2dst_pps'], row['dst2src_pps'], row['exp_id']
    plt.xticks(range(1,6))
    plt.scatter(x, y1, c='red', marker='s', s=PLOT_CONFIG['marker_size'])
    plt.scatter(x, y2, c='blue', marker='o', s=PLOT_CONFIG['marker_size'])
    # plt.annotate(label, (x, y1), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

plt.legend(['client-to-server', 'server-to-client'], fontsize=PLOT_CONFIG['legend_size'])
plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}PPS_vs_ACR.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.xlabel('MOS', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('Packet Size', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('MOS vs. Average Packet Size', fontsize=PLOT_CONFIG['title_size'])

for i, row in analysis_df.iterrows():
    x, y1, y2, label = row['self_rate_qoe_score'], row['src2dst_avg_pkt_size'], row['dst2src_avg_pkt_size'], row['exp_id']
    plt.scatter(x, y1, c='red', marker='s', s=PLOT_CONFIG['marker_size'])
    plt.scatter(x, y2, c='blue', marker='o', s=PLOT_CONFIG['marker_size'])
    plt.annotate(label, (x, y1), textcoords="offset points", xytext=(0, 0), ha='center', c='r')
    plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='b')

plt.legend(['client-to-server', 'server-to-client'], fontsize=PLOT_CONFIG['legend_size'])
plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}MOS_vs_Average_Packet_Size.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
len(analysis_df)

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.xlabel('src2dst_avg_pkt_size', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('dst2src_avg_pkt_size', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('Median PPS', fontsize=PLOT_CONFIG['title_size'])

for i, row in analysis_df.iterrows():
    x, y, label1 = row['src2dst_avg_pkt_size'], row['dst2src_avg_pkt_size'], row['ping']
    plt.scatter(x, y, c='blue', s=PLOT_CONFIG['marker_size'], alpha=0.7)
    plt.annotate(label1, (x, y), textcoords="offset points", xytext=(0, 0), ha='center', c='b')
    # plt.annotate(label, (x, y2), textcoords="offset points", xytext=(0, 0), ha='center', c='r')

plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}Client_to_Server_vs_Server_to_Client_Packet_Size.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
plt.figure(figsize=PLOT_CONFIG['figure_size'])

plt.xlabel('self_rate_qoe_score', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('ping', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('self_rate_qoe_score vs ping', fontsize=PLOT_CONFIG['title_size'])

plt.scatter(df['self_rate_qoe_score'], df['ping'], s=PLOT_CONFIG['marker_size'])
plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}Self_Rate_QoE_vs_Ping.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle

# Load the dataset
features = ['src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median', 'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max', 'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median', 'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max', 'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median', 'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max', 'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median', 'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max', 'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median', 'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max']
dataset = pd.read_csv('/Users/<user>/PhD/github/qoe-research/csv/window.csv') 
dataset.dropna(subset=features, inplace=True)
dataset['y_qoe_by_ping'] = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
X_train = dataset[dataset['exp_id'] < 59][features]
y_train_by_ping = dataset[dataset['exp_id'] < 59]['y_qoe_by_ping']
X_test = dataset[dataset['exp_id'] >= 59][features]
y_test_by_ping = dataset[dataset['exp_id'] >= 59]['y_qoe_by_ping']

# X = dataset[features]
# y_qoe_by_ping = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
# X_train, X_test, y_train_by_ping, y_test_by_ping = train_test_split(X, y_qoe_by_ping, test_size=0.4, random_state=42)
y_train_by_survey = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_test_by_survey = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_train_by_impact = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
y_test_by_impact = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle

# Load the dataset
features = ['src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median', 'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max', 'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median', 'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max', 'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median', 'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max', 'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median', 'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max', 'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median', 'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max']
dataset = pd.read_csv('/Users/<user>/PhD/github/qoe-research/csv_5/window.csv') 
dataset.dropna(subset=features, inplace=True)
X = dataset[features]
y_qoe_by_ping = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
X_train, X_test, y_train_by_ping, y_test_by_ping = train_test_split(X, y_qoe_by_ping, test_size=0.4, random_state=42)
y_train_by_survey = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_test_by_survey = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
# self_rate_player_performance	
y_train_by_performance = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_player_performance']
y_test_by_performance = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_player_performance']
# y_train_by_impact = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
# y_test_by_impact = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])

In [ ]:
len(X_train), len(y_test_by_ping), len(X_train) + len(X_test)

In [ ]:
# have both y_qoe_by_ping.value_counts(), y_test_by_ping.value_counts() in one dataframe
ping_df = pd.DataFrame({'y_qoe_by_ping': y_qoe_by_ping.value_counts(), 'y_test_by_ping': y_test_by_ping.value_counts()})
qoe_df = pd.DataFrame({'y_qoe_by_survey': y_train_by_survey.value_counts(), 'y_test_by_survey': y_test_by_survey.value_counts()})

In [ ]:
ping_df

In [ ]:
ping_df

In [ ]:
qoe_df

In [ ]:
ping_clf = RandomForestClassifier()
survey_clf = RandomForestClassifier()
performance_clf = RandomForestClassifier()
# Train the models
ping_clf.fit(X_train, y_train_by_ping)
survey_clf.fit(X_train, y_train_by_survey)
performance_clf.fit(X_train, y_train_by_performance)

y_pred_by_ping = ping_clf.predict(X_test)
y_pred_by_survey = survey_clf.predict(X_test)
y_pred_by_performance = performance_clf.predict(X_test)


In [ ]:
X_train.iloc[1079]

In [ ]:
# print(len(X), len(X_train), len(X_test))
# print([v for v in y_pred])
# print(list(y_test))
y_test_list = list(y_pred_by_ping)
for i, _ in enumerate(y_test_list):
    print(y_test_list[i], y_pred_by_ping[i], y_pred_by_survey[i])
    # print(y_test_list[i], y_pred_qoe_by_ping[i], y_pred_qoe_by_survey[i])

# with open('model4.pkl', 'wb') as file:
#     pickle.dump(clf, file) 

In [ ]:
plot_confusion_matrix(y_test_by_performance, y_pred_by_performance)

In [ ]:
plot_confusion_matrix(y_test_by_survey, y_pred_by_survey)

In [ ]:
confusion_matrix(y_test_by_ping, y_pred_by_ping, normalize='pred')

In [ ]:

plot_confusion_matrix(y_test_by_ping, y_pred_by_ping)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np
import seaborn as sns

def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    true_labels = ['Bad', 'Poor', 'Fair', 'Good', 'Excellent']
    # true_labels = ['Excellent', 'Good', 'Fair', 'Bad']
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=true_labels,
                yticklabels=true_labels)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    for i, data in enumerate(cm):
        plt.bar(range(len(data)), data)

        # Set labels and title
        plt.xlabel('Category')
        plt.xticks(range(len(true_labels)), true_labels)
        plt.ylabel('Frequency')
        plt.title(f'True Label {true_labels[i]} Predications')

        # Show the plot
        plt.show()

In [ ]:

dir_name = '/Users/<user>/PhD/github/qoe-research/eyal-csv'
# filename = '20230611-HUMAN5v5_FPS60_#1_D_0_0_50_U_0_0_50.csv'
for filename in os.listdir(dir_name):
    df = pd.read_csv(os.path.join(dir_name, filename))
    df = df[df['pkt_dir_ratio'] > 0.1]

    # Create some mock data
    t = df['pktIdx']
    data1 = df['src2dst_pps'].rolling(window=10).max()
    data2 = df['dst2src_pps'].rolling(window=10).max()

    fig, ax1 = plt.subplots()


    color = 'tab:red'
    ax1.set_xlabel('time (s)')
    ax1.set_ylabel('src2dst_pps', color=color)
    ax1.plot(t, data1, color=color)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

    color = 'tab:blue'
    ax2.set_ylabel('dst2src_pps', color=color)  # we already handled the x-label with ax1
    ax2.plot(t, data2, color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    plt.xticks(range(0,round(max(t)), 10),rotation=45)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    # plt.show()
    fig.savefig(os.path.join('/Users/<user>/PhD/qoe/short_max_png', filename.replace('.csv', '.png')))   # save the figure to file
    plt.close(fig)    # close the figure window

In [ ]:
filepath = '/Users/<user>/PhD/github/qoe-research/beni-csv/window-0.csv'
for y1, y2 in [
    ('src2dst_avg_pkt_size_max', 'dst2src_avg_pkt_size_max'),
    ('src2dst_avg_pkt_size_min', 'dst2src_avg_pkt_size_min'),
    ('src2dst_avg_pkt_size_mean', 'dst2src_avg_pkt_size_mean'),
    ('src2dst_pps_max', 'dst2src_pps_max'),
    ('src2dst_pps_min', 'dst2src_pps_min'),
    ('src2dst_pps_mean', 'dst2src_pps_mean')
]:
    df = pd.read_csv(filepath)
    df = df[df['pkt_dir_ratio'] > 0.1]
    t = df['pktIdx'] * 2
    data1 = df[y1]
    data2 = df[y2]

    fig, ax1 = plt.subplots(figsize=(10, 2))  # set the figsize parameter to (10, 5) for a wider plot

    color = 'tab:red'
    ax1.set_xlabel('time (s)')
    ax1.set_ylabel(y1, color=color)
    ax1.plot(t, data1, color=color)
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

    color = 'tab:blue'
    ax2.set_ylabel(y2, color=color)  # we already handled the x-label with ax1
    ax2.plot(t, data2, color=color)
    ax2.tick_params(axis='y', labelcolor=color)

    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    plt.show()
    fig.savefig(os.path.join('/Users/<user>/PhD/github/qoe-research/png', y1.replace('src2dst_', '') + '.png'))   # save the figure to file
    plt.close(fig)    # close the figure window


In [ ]:
df = pd.read_csv('/Users/<user>/PhD/github/qoe-research/short/1.csv')
df = df[df['pkt_dir_ratio'] > 0.1]
t = df['pktIdx']
data1 = df['src2dst_avg_pkt_size']
data2 = df['dst2src_avg_pkt_size']

fig, ax1 = plt.subplots(figsize=(10, 2))  # set the figsize parameter to (10, 5) for a wider plot

color = 'tab:red'
ax1.set_xlabel('time (s)')
ax1.set_ylabel('src2dst_avg_pkt_size', color=color)
ax1.plot(t, data1, color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('dst2src_avg_pkt_size', color=color)  # we already handled the x-label with ax1
ax2.plot(t, data2, color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()
plt.savefig('figure_name.png', dpi=300, bbox_inches='tight')
plt.close()  # Close the figure to free memory

In [ ]:
df = pd.read_csv('/Users/<user>/PhD/github/qoe-research/band/window.csv')

In [ ]:
exp7_df = df[df['exp_id'] == 7]

In [ ]:
# plot a trend line chart of src2dst_pps in red and dst2src_pps in blue vs. pktIdx starting for pktIdx 25 to 300
# make src2dst_pps dotted line
# make dst2src_pps solid line
# exp7_df.plot(x='pktIdx', y=['src2dst_pps_median', 'dst2src_pps_median'], color=['red', 'blue'], xlim=(25, 301), style=['o', 's'])

plt.rcParams.update({'font.size': PLOT_CONFIG['base_font_size']})
exp7_df.plot(x='pktIdx', y=['src2dst_pps', 'dst2src_pps'], color=['red', 'blue'], xlim=(25, 301), style=['o', 's'], figsize=PLOT_CONFIG['figure_size'])
# y max value should be 200
plt.ylim(0, 200)
plt.legend().remove()
plt.xticks(range(25, 301, 25))
plt.xlabel('Gameplay Time', fontsize=PLOT_CONFIG['axis_label_size'])
plt.ylabel('PPS', fontsize=PLOT_CONFIG['axis_label_size'])
# plt.title('PPS vs. Gameplay Time', fontsize=PLOT_CONFIG['title_size'])
plt.legend(['client-to-server', 'server-to-client'], fontsize=PLOT_CONFIG['legend_size'])
plt.grid(True, which='both', linestyle='--', alpha=PLOT_CONFIG['grid_alpha'])
plt.tick_params(axis='both', which='major', labelsize=PLOT_CONFIG['tick_label_size'])
plt.tight_layout()
plt.savefig(f"{PLOT_CONFIG['output_dir']}PPS_vs_Gameplay_Time_Limited.png", dpi=PLOT_CONFIG['dpi'], bbox_inches='tight')
plt.close()